# Event-level histograms and overweight protection

Plot the weighted and unweighted event-level $\hat{p}_{T}$ and vertex-z histograms produced by `processForestSimple.C`. The final sections inspect the gen/reco dijet $p_{T}^{ave}/\hat{p}_{T}$ maps, reproduce the discrete upper-tail overweight-protection threshold from `macro/plotMcClosures.C::plotOverweightProtection`, fit that threshold, and overlay the fits on the original maps.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path('/Users/gnigmat/work/cms/jetAnalysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.python.event_plots import (
    draw_event_histograms, draw_overweight_map, draw_overweight_protection,
)
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_direction_file,
)

## Configuration

The defaults compare p-going, Pb-going, and the existing combined file for Pythia. `FILE_STEM='jetId'` matches the current production outputs. Integral normalization makes direction-shape comparisons meaningful; use `'none'` to retain the stored weighted or unweighted yields. Weighted overlays intentionally disable grids, while `DRAW_GRID` controls both axes for the unweighted overlays and overweight-protection plots. `SAVE_PNG` optionally writes a PNG beside each PDF. `OVERWEIGHT_DIRECTION` independently selects the file used for the gen/reco diagnostic.

In [ ]:
GENERATOR = 'pythia'  # embedding or pythia
DIRECTIONS = ('pgoing', 'Pbgoing', 'combined')
FILE_STEM = 'jetId'
NORMALIZATION = 'integral'  # none or integral
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'events'
DRAW_GRID = True
SAVE_PNG = False
CUT_FRACTION = 0.0001  # upper 0.01% tail
OVERWEIGHT_DIRECTION = 'combined'
DIRECTION_COMPARISON_TAG = 'directionComparison'

def mc_file(direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
    return resolve_direction_file(BASE_DIR, GENERATOR, direction, FILE_STEM)

files = {direction: mc_file(direction) for direction in DIRECTIONS}
missing = [str(path) for path in files.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing configured ROOT files:\n' + '\n'.join(missing))
files

## Weighted event distributions

In [ ]:
weighted_canvases = {}
for key, x_title, log_y in (
    ('hPtHat', '#hat{p}_{T} (GeV)', True),
    ('hVz', 'v_{z} (cm)', False),
):
    histograms = {direction: load_histogram(str(path), key)
                  for direction, path in files.items()}
    canvas, plotted = draw_event_histograms(
        histograms, title=f'{GENERATOR}: {key}', x_title=x_title,
        normalization=NORMALIZATION, log_y=log_y, grid=False,
        output=OUTPUT_DIR / f'{GENERATOR}_{DIRECTION_COMPARISON_TAG}_{key}.pdf',
        save_png=SAVE_PNG,
    )
    weighted_canvases[key] = (canvas, plotted)
    display(canvas)

## Unweighted event distributions

These use `hPtHatUnweighted` and `hVzUnweighted`, which contain event populations before cross-section/event weighting. The configured display normalization is still applied to the plotted clones.

In [ ]:
unweighted_canvases = {}
for key, x_title, log_y in (
    ('hPtHatUnweighted', '#hat{p}_{T} (GeV)', True),
    ('hVzUnweighted', 'v_{z} (cm)', False),
):
    histograms = {direction: load_histogram(str(path), key)
                  for direction, path in files.items()}
    canvas, plotted = draw_event_histograms(
        histograms, title=f'{GENERATOR}: {key}', x_title=x_title,
        normalization=NORMALIZATION, log_y=log_y, grid=DRAW_GRID,
        output=OUTPUT_DIR / f'{GENERATOR}_{DIRECTION_COMPARISON_TAG}_{key}.pdf',
        save_png=SAVE_PNG,
    )
    unweighted_canvases[key] = (canvas, plotted)
    display(canvas)

## Overweight-protection inputs

The two-dimensional distributions expose unusually large dijet momentum relative to the hard-scattering scale. The macro's threshold calculation excludes underflow/overflow and scans each Y projection from high to low until it accumulates `CUT_FRACTION` of the in-range weight.

In [ ]:
overweight_file = mc_file(OVERWEIGHT_DIRECTION)
overweight_histograms = {
    level: load_histogram(str(overweight_file), f'h{level}DijetPtAveOverPtHatVsPtHat')
    for level in ('Gen', 'Reco')
}

overweight_maps = {}
for level, histogram in overweight_histograms.items():
    canvas = draw_overweight_map(
        histogram, title=f'{GENERATOR} {OVERWEIGHT_DIRECTION}: {level} dijets',
        grid=DRAW_GRID,
        output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_{level.lower()}_overweight_map.pdf',
        save_png=SAVE_PNG,
    )
    overweight_maps[level] = canvas
    display(canvas)

## Upper-tail thresholds and fits

The points use the macro's discrete Y-bin-center threshold definition. Both curves use $p_{0}+p_{1}\exp(-p_{2}x)+p_{3}\exp(-p_{4}x)$ over 15–950 GeV. Empty $\hat{p}_{T}$ bins remain zero; ROOT excludes points outside the fit range and bins without usable uncertainties from the fit. The cell prints every fitted parameter and $\chi^{2}/\mathrm{NDF}$.

In [ ]:
threshold_canvas, thresholds, fits = draw_overweight_protection(
    overweight_histograms['Gen'], overweight_histograms['Reco'],
    cut_fraction=CUT_FRACTION, grid=DRAW_GRID,
    output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_overweight_threshold.pdf',
    save_png=SAVE_PNG,
)
for level, fit in fits.items():
    print(f'{level} fit:')
    for index in range(fit.GetNpar()):
        print(f'  p{index} = {fit.GetParameter(index):.8g} +/- {fit.GetParError(index):.3g}')
    chi2 = fit.GetChisquare()
    ndf = fit.GetNDF()
    chi2_ndf = chi2 / ndf if ndf > 0 else float('nan')
    print(f'  chi2 = {chi2:.6g}, NDF = {ndf}, chi2/NDF = {chi2_ndf:.6g}')
display(threshold_canvas)

## Gen overweight map with fitted protection threshold

The fitted threshold is overlaid on the original two-dimensional gen distribution.

In [ ]:
gen_fit_overlay_canvas = draw_overweight_map(
    overweight_histograms['Gen'],
    title=f'{GENERATOR} {OVERWEIGHT_DIRECTION}: Gen dijets with threshold fit',
    fit=fits['Gen'], grid=DRAW_GRID,
    output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_gen_overweight_map_fit.pdf',
    save_png=SAVE_PNG,
)
display(gen_fit_overlay_canvas)

## Reco overweight map with fitted protection threshold

The fitted threshold is overlaid on the original two-dimensional reco distribution.

In [ ]:
reco_fit_overlay_canvas = draw_overweight_map(
    overweight_histograms['Reco'],
    title=f'{GENERATOR} {OVERWEIGHT_DIRECTION}: Reco dijets with threshold fit',
    fit=fits['Reco'], grid=DRAW_GRID,
    output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_reco_overweight_map_fit.pdf',
    save_png=SAVE_PNG,
)
display(reco_fit_overlay_canvas)